# PGD — Towards Deep Learning Models Resistant to Adversarial Attacks (2018)

_Papers · Meta-learning, Bayesian & Robustness_

**Frame robustness as a min-max game; attack with multi-step Projected Gradient Descent and train on those examples.**

---

This notebook builds the PGD attack and adversarial-training loop from scratch, one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

We build the whole pipeline in five steps: (0) hand-check one PGD step on a scalar so the projection is concrete, (1) make a hard "moons" dataset, (2) define the model and the FGSM / PGD attacks, (3) train a normal and an adversarial model with the min-max loop, (4) evaluate both under clean / FGSM / PGD.

### Step 0 — Sanity-check one PGD step

Before touching a network, we trace a single PGD step on a scalar logit $w\cdot x$ with $w=2$ and target label $1$. We take the gradient of the BCE loss w.r.t. the input, step in the *sign* direction by $\alpha$, then **project** back into the allowed $\epsilon$-ball $[x_0-\epsilon,\,x_0+\epsilon]$. That clip is the heart of PGD — it keeps the perturbation within budget.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# logit = w*x, w=2.0, label y=1, BCE loss; x0=0.5, alpha=0.20, eps=0.15 -> ball [0.35, 0.65].
x0 = torch.tensor(0.5)
w = torch.tensor(2.0)
eps_we = 0.15
alpha_we = 0.20

xt = x0.clone().requires_grad_(True)
loss_we = F.binary_cross_entropy_with_logits(w * xt, torch.tensor(1.0))
g = torch.autograd.grad(loss_we, xt)[0]

ascend = x0.item() + alpha_we * torch.sign(g).item()              # one sign-step uphill
lower = x0.item() - eps_we
upper = x0.item() + eps_we
proj = min(max(ascend, lower), upper)                             # clip into the eps-ball

print("worked example: sign(grad)=%.0f  ascend=%.2f  project=%.2f  (ball [%.2f, %.2f])" % (
      torch.sign(g).item(), ascend, proj, lower, upper))
# worked example: sign(grad)=-1  ascend=0.30  project=0.35  (ball [0.35, 0.65])

### Step 1 — Build a hard two-class dataset

Adversarial attacks only bite on a model that has a non-trivial decision boundary, so we use interleaving "moons" with noise. Each `moons` call returns a shuffled set of points and $\{0,1\}$ labels as tensors. We make a train and a test split with different seeds.

In [ ]:
# A hard two-class toy dataset: interleaving "moons" (so attacks actually bite).
def moons(n, noise=0.25, seed=0):
    rng = np.random.RandomState(seed)
    h = n // 2
    t = np.linspace(0, np.pi, h)
    outer = np.stack([np.cos(t), np.sin(t)], 1)
    inner = np.stack([1 - np.cos(t), 1 - np.sin(t) - 0.5], 1)
    X = np.concatenate([outer, inner])
    y = np.concatenate([np.zeros(h), np.ones(h)])
    X += rng.normal(scale=noise, size=X.shape)
    p = rng.permutation(n)
    return torch.tensor(X[p], dtype=torch.float32), torch.tensor(y[p], dtype=torch.float32)


Xtr, ytr = moons(600, seed=1)
Xte, yte = moons(400, seed=2)

### Step 2 — Define the model and the attacks

The model is a small MLP. The two attacks share a goal — push the input to raise the loss — but differ in steps. **FGSM** takes one sign-step of size $\epsilon$. **PGD** starts at a random point in the ball and takes many smaller $\alpha$-steps, re-projecting into the $\epsilon$-ball after each one. `acc` just measures accuracy with gradients off.

In [ ]:
# The model, composed with torch.nn.
def make_net():
    return nn.Sequential(
        nn.Linear(2, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, 1),
    )


loss_fn = nn.BCEWithLogitsLoss()
EPS = 0.6        # L-inf radius (the perturbation budget)
ALPHA = 0.15     # PGD step size
PGD_STEPS = 10   # PGD iterations


# The NOVEL attacks, built by hand.
def fgsm(net, X, y, eps):
    Xa = X.clone().requires_grad_(True)
    l = loss_fn(net(Xa).squeeze(1), y)
    g, = torch.autograd.grad(l, Xa)                  # gradient w.r.t. the INPUT
    return (X + eps * torch.sign(g)).detach()        # one sign-step, raises the loss


def pgd(net, X, y, eps, alpha, steps):
    Xa = (X + torch.empty_like(X).uniform_(-eps, eps)).detach()   # random start in the ball
    for _ in range(steps):
        Xa.requires_grad_(True)
        l = loss_fn(net(Xa).squeeze(1), y)
        g, = torch.autograd.grad(l, Xa)
        Xa = Xa.detach() + alpha * torch.sign(g)                  # ascend the loss
        Xa = torch.min(torch.max(Xa, X - eps), X + eps).detach()  # Proj: clip into eps-ball
    return Xa


def acc(net, X, y):
    with torch.no_grad():
        return ((net(X).squeeze(1) > 0).float() == y).float().mean().item()

### Step 3 — Train normal vs adversarial (the min-max loop)

The paper's objective (Eqn. 2.1) is a min-max: the **inner max** finds the worst perturbation (PGD), the **outer min** updates the weights against it. Normal training skips the inner max and trains on clean inputs. We train both from the same seed so the only difference is the adversarial batch.

In [ ]:
# Training: normal vs the MIN-MAX adversarial loop (Eqn. 2.1).
def train(net, adversarial):
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for ep in range(150):
        if adversarial:
            Xb = pgd(net, Xtr, ytr, EPS, ALPHA, PGD_STEPS)   # inner max: worst-case inputs
        else:
            Xb = Xtr                                          # normal training: clean inputs
        opt.zero_grad()
        loss = loss_fn(net(Xb).squeeze(1), ytr)              # outer min: fit the weights
        loss.backward()
        opt.step()
    return net


torch.manual_seed(1)
normal = train(make_net(), adversarial=False)
torch.manual_seed(1)
robust = train(make_net(), adversarial=True)

### Step 4 — Evaluate under clean / FGSM / PGD

Now we score both models three ways. The normal model is accurate on clean data but collapses under attack — and collapses *more* under PGD than FGSM, confirming PGD is the stronger attack. The adversarially-trained model gives up some clean accuracy but holds up far better under PGD. (Small run; not the paper's exact numbers.)

In [ ]:
# Evaluate: clean vs FGSM vs PGD, for both models.
def report(name, net):
    c = acc(net, Xte, yte)
    f = acc(net, fgsm(net, Xte, yte, EPS), yte)
    p = acc(net, pgd(net, Xte, yte, EPS, ALPHA, PGD_STEPS), yte)
    print("%-20s clean=%.3f  FGSM=%.3f  PGD=%.3f" % (name, c, f, p))


print("eps=%.2f (L-inf), PGD %d steps alpha=%.2f, test n=%d" % (EPS, PGD_STEPS, ALPHA, len(yte)))
report("Normally trained", normal)
report("PGD adv-trained", robust)
# Normally trained     clean=0.957  FGSM=0.228  PGD=0.153   <- PGD is the stronger attack
# PGD adv-trained      clean=0.777  FGSM=0.522  PGD=0.482   <- robust acc up 0.153 -> 0.482
# (Our small run, not the paper's reported numbers. Exact values vary by seed/hardware.)

## Visualize the data & results

_Under the same epsilon-budget, how do clean / FGSM / PGD accuracy compare for a normally-trained model versus a PGD-adversarially-trained model?_

### Step — Rebuild the pipeline compactly

This visualization panel is self-contained: it re-imports, re-seeds, and redefines the dataset, model, and both attacks so it can run on its own. The logic is identical to the reference implementation above — same moons data, same FGSM / PGD, same hyperparameters.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)


def moons(n, noise=0.25, seed=0):
    rng = np.random.RandomState(seed)
    h = n // 2
    t = np.linspace(0, np.pi, h)
    outer = np.stack([np.cos(t), np.sin(t)], 1)
    inner = np.stack([1 - np.cos(t), 1 - np.sin(t) - 0.5], 1)
    X = np.concatenate([outer, inner])
    y = np.concatenate([np.zeros(h), np.ones(h)])
    X += rng.normal(scale=noise, size=X.shape)
    p = rng.permutation(n)
    return torch.tensor(X[p], dtype=torch.float32), torch.tensor(y[p], dtype=torch.float32)


Xtr, ytr = moons(600, seed=1)
Xte, yte = moons(400, seed=2)


def make_net():
    return nn.Sequential(
        nn.Linear(2, 64), nn.ReLU(),
        nn.Linear(64, 64), nn.ReLU(),
        nn.Linear(64, 1),
    )


lf = nn.BCEWithLogitsLoss()
EPS = 0.6
ALPHA = 0.15
STEPS = 10


def fgsm(net, X, y):
    Xa = X.clone().requires_grad_(True)
    g, = torch.autograd.grad(lf(net(Xa).squeeze(1), y), Xa)
    return (X + EPS * torch.sign(g)).detach()


def pgd(net, X, y):
    Xa = (X + torch.empty_like(X).uniform_(-EPS, EPS)).detach()
    for _ in range(STEPS):
        Xa.requires_grad_(True)
        g, = torch.autograd.grad(lf(net(Xa).squeeze(1), y), Xa)
        Xa = Xa.detach() + ALPHA * torch.sign(g)
        Xa = torch.min(torch.max(Xa, X - EPS), X + EPS).detach()   # projection onto eps-ball
    return Xa


def acc(net, X, y):
    with torch.no_grad():
        return ((net(X).squeeze(1) > 0).float() == y).float().mean().item()

### Step — Train both models and compare the three accuracies

With the pieces in place, we train the normal and robust models from the same seed and print the clean / FGSM / PGD accuracy triple for each. The numbers reproduce the earlier finding: PGD is the harder attack, and adversarial training trades clean accuracy for a large gain in PGD-robust accuracy.

In [ ]:
def train(net, adv):
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    for ep in range(150):
        Xb = pgd(net, Xtr, ytr) if adv else Xtr
        opt.zero_grad()
        lf(net(Xb).squeeze(1), ytr).backward()
        opt.step()
    return net


torch.manual_seed(1)
normal = train(make_net(), False)
torch.manual_seed(1)
robust = train(make_net(), True)

for name, net in [("normal", normal), ("robust", robust)]:
    clean = round(acc(net, Xte, yte), 3)
    f = round(acc(net, fgsm(net, Xte, yte), yte), 3)
    p = round(acc(net, pgd(net, Xte, yte), yte), 3)
    print(name, [clean, f, p])
# normal [0.957, 0.228, 0.153]
# robust [0.777, 0.522, 0.482]
# PGD (0.153) beats FGSM (0.228) as an attack; adv-training lifts PGD acc 0.153 -> 0.482.
# Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The projection is the whole game. Take the worked example but remove the projection: start at
            $x^0=0.50$, $\alpha=0.20$, $\epsilon=0.15$, and run three ascent steps with sign $-1$ each
            time, never clipping. Where does $x$ end up, and why does that break the attack's definition?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Step without projecting: $0.50 \to 0.30 \to 0.10 \to -0.10$ (subtract $0.20$ three times). — _Each step subtracts $\alpha\cdot 1$; with no clip nothing stops it._
- Compare to the ball $[0.35, 0.65]$: the final $-0.10$ is far outside the allowed budget $\epsilon=0.15$. — _The perturbation $|x - x^0| = 0.60$ is $4\times$ the budget &mdash; no longer "imperceptible"._
- Now add the projection back: every step clips to $[0.35, 0.65]$, so the iterate is $0.50 \to 0.35 \to 0.35 \to 0.35$. — _Projection caps the perturbation at $\epsilon$; the iterate sits at the ball's edge, which is the constrained maximizer here._

**Answer:** Without projection the input runs off to $-0.10$, a perturbation of $0.60$ &mdash; four times the
                 budget $\epsilon=0.15$. That violates the inner problem $\max_{\delta\in S}L$, which only allows
                 $|\delta|\le\epsilon$. With projection, every step clips back to $[0.35,0.65]$, pinning the
                 iterate at the edge $0.35$. The projection is exactly what keeps PGD solving the
                 constrained maximization instead of unconstrained ascent.

</details>

**Problem 2.** Why is PGD stronger than single-step FGSM? Both stay inside the same $\epsilon$-ball, so why
            does the multi-step attack lower a normal model's accuracy more?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write FGSM: one step of size $\epsilon$ using only the sign of $\nabla_x L$ at the clean point $x$. — _It commits to a single straight-line direction based on the gradient at one point._
- Write PGD: many steps of size $\alpha\lt\epsilon$, recomputing $\nabla_x L$ at each new iterate and re-projecting. — _The loss surface is curved; the best direction changes as you move. PGD re-aims each step._
- Conclude: PGD follows the curved loss surface to a higher-loss corner of the ball than FGSM's single straight shot reaches. — _More gradient evaluations inside the ball find a worse-case input; that is why our run shows normal-model accuracy lower under PGD than under FGSM._

**Answer:** FGSM takes one straight step using the gradient at the clean input. The loss surface is curved, so
                 that single direction is only locally best. PGD takes many smaller steps, recomputing the gradient
                 and re-projecting each time, so it tracks the curvature to a higher-loss point inside the same
                 ball. Same budget, better search &mdash; a stronger attack. In our run the normally-trained model
                 drops to about 0.23 under FGSM but 0.15 under PGD.

</details>

**Problem 3.** The ablation: turn off adversarial training. You have a working min-max loop. Replace the PGD
            inner step in training with nothing (train on clean inputs only), everything else identical. What
            happens to clean accuracy and to PGD accuracy, and what trade-off does this expose?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove the inner max: in each training step, drop the PGD call and use the clean batch. — _This collapses Eqn. 2.1's min-max back to ordinary empirical-risk minimization &mdash; it is the normal-training baseline._
- Re-evaluate under PGD: the normally-trained model collapses (our run: ~0.15) while the adversarially-trained one holds far higher (~0.48). — _Without seeing worst-case inputs in training, the model has no incentive to be flat in the $\epsilon$-ball._
- Check clean accuracy: the normal model is higher on clean data (~0.96) than the robust one (~0.78). — _Adversarial training spends capacity on robustness, which costs clean accuracy &mdash; the robustness/accuracy trade-off._

**Answer:** Dropping the PGD inner step turns the min-max objective back into normal training. That model wins
                 on clean data (our run: ~0.96 vs ~0.78 for the robust model) but collapses under PGD
                 (~0.15 vs ~0.48). The ablation exposes the central trade-off: PGD adversarial training buys robust
                 accuracy at the cost of some clean accuracy. The inner maximization is what forces the network to
                 be flat in a small ball around each point. (Our small run, not the paper's numbers.)

</details>